# Notebook 1: Data Collection from PDB

This notebook downloads protein structures from the RCSB Protein Data Bank,
runs DSSP to assign secondary structure labels, and saves a clean dataset.

### Design decisions
- **Source:** X-ray crystallography structures only (highest resolution)
- **Diversity:** Sampled from PDB's own 30% sequence identity clusters. One representative per cluster ensures proteins in the dataset are genuinely different from each other
- **One chain per entry:** Only the longest valid chain is kept per PDB entry, preventing multi-chain complexes from dominating the dataset
- **3-state labels:** DSSP's 8-state scheme (H, G, I, E, B, T, S, C) is mapped to 3 states: Helix (H), Strand (E), Coil (C)


In [ ]:
import os
import random
import requests
import pandas as pd
from Bio.PDB import PDBParser, DSSP


download_folder = "pdb_files3"
output_csv = "protein_sequences_ss_3.csv"
num_proteins = 5000

os.makedirs(download_folder, exist_ok=True)


# RCSB Query — using PDB's pre-clustered 30% identity representatives
r = requests.get("https://cdn.rcsb.org/resources/sequence/clusters/clusters-by-entity-30.txt")
lines = r.text.strip().split("\n")
print(f"Total 30% clusters in PDB: {len(lines)}")

random.seed(42)
sampled_lines = random.sample(lines, min(num_proteins, len(lines)))

pdb_ids = []
for line in sampled_lines:
    rep = line.strip().split()[0]   # first member = representative
    pdb_id = rep.split("_")[0]      # "101M_1" → "101M"
    if pdb_id.startswith("AF"):     # skip AlphaFold entries
        continue
    pdb_ids.append(pdb_id)

# Remove duplicates (different entities can share a PDB ID)
pdb_ids = list(dict.fromkeys(pdb_ids))
print(f"Fetching {len(pdb_ids)} representative PDB entries.")


# Processing
parser = PDBParser(QUIET=True)
data = []
seen_sequences = set()

MIN_LEN = 50
MAX_LEN = 1000

for pdb_id in pdb_ids:
    pdb_file = os.path.join(download_folder, f"{pdb_id}.pdb")

   
    # Download PDB
    if not os.path.exists(pdb_file):
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        resp = requests.get(url)
        if resp.status_code == 200:
            with open(pdb_file, "w") as f:
                f.write(resp.text)
            print(f"Downloaded {pdb_id}")
        else:
            print(f"Failed to download {pdb_id}")
            continue

    try:
        structure = parser.get_structure(pdb_id, pdb_file)
        model = structure[0]

        
        # Run DSSP once per structure
        dssp = DSSP(model, pdb_file, dssp="mkdssp")


        # Pick ONE chain per PDB entry — the longest valid one
        best_chain = None  # will hold the best (sequence, ss, chain_id) tuple

        for chain in model:
            chain_id = chain.id

            keys = [k for k in dssp.keys() if k[0] == chain_id]
            keys = sorted(keys, key=lambda x: x[1])

            if len(keys) == 0:
                continue

            sequence = "".join([dssp[k][1] for k in keys])
            ss_full  = "".join([dssp[k][2] for k in keys])

            # Remove invalid amino acids
            valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
            filtered_seq = ""
            filtered_ss  = ""
            for aa, s in zip(sequence, ss_full):
                if aa in valid_aa:
                    filtered_seq += aa
                    filtered_ss  += s

            sequence = filtered_seq
            ss_full  = filtered_ss

            if len(sequence) == 0:
                continue
            if len(sequence) < MIN_LEN or len(sequence) > MAX_LEN:
                continue

            # Convert 8-state → 3-state
            ss_simple = "".join(
                'H' if s in "HGI" else 'E' if s in "EB" else 'C'
                for s in ss_full
            )

            if len(sequence) != len(ss_simple):
                continue

            # Keep the longest valid chain for this PDB entry
            if best_chain is None or len(sequence) > len(best_chain[0]):
                best_chain = (sequence, ss_simple, chain_id)


        # Save the single best chain (if any)
        if best_chain is None:
            print(f"No valid chain found for {pdb_id}")
            continue

        sequence, ss_simple, chain_id = best_chain
        full_id = f"{pdb_id}_{chain_id}"

        # Skip duplicate sequences across different PDB entries
        if sequence in seen_sequences:
            print(f"Duplicate sequence, skipping {full_id}")
            continue

        seen_sequences.add(sequence)

        data.append({
            "pdb_id": full_id,
            "sequence": sequence,
            "secondary_structure": ss_simple,
            "length": len(sequence)
        })

        print(f"Processed {full_id} — {len(sequence)} residues")

    except Exception as e:
        print(f"Failed {pdb_id}: {e}")



df = pd.DataFrame(data)
df.to_csv(output_csv, index=False)

print(f"\nSaved {len(df)} clean chains → {output_csv}")

Total 30% clusters in PDB: 258303
Fetching 744 representative PDB entries.
Downloaded 1NOF
Processed 1NOF_A — 383 residues
Downloaded 3RC3


/home/mehrak/bioenv/lib/python3.12/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Resulting mmCIF file is not valid!
Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Warning, the input file is not valid. Run with --verbose to see why.

  warnings.warn(err)


Processed 3RC3_A — 593 residues
Downloaded 7ORH
Processed 7ORH_A — 228 residues
Downloaded 4DOH


/home/mehrak/bioenv/lib/python3.12/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 4 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 4 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Resulting mmCIF file is not valid!
Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 4 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 4 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Warning, the input file is not valid. Run with --verbose to see why.

  warnings.warn(err)


Processed 4DOH_R — 204 residues
Failed to download MA
Downloaded 1HJ1


...

Processed 4I1M_C — 293 residues
Downloaded 7LGL
No valid chain found for 7LGL
Downloaded 5YXU


/home/mehrak/bioenv/lib/python3.12/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Resulting mmCIF file is not valid!
Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Warning, the input file is not valid. Run with --verbose to see why.

  warnings.warn(err)


Processed 5YXU_E — 276 residues
Downloaded 8D05


/home/mehrak/bioenv/lib/python3.12/site-packages/Bio/PDB/DSSP.py:199: UserWarning: Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Resulting mmCIF file is not valid!
Links for struct_ref_seq:pdbx_poly_seq_scheme:3 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Links for struct_ref_seq:pdbx_poly_seq_scheme:4 are incomplete
  There are 1 items in struct_ref_seq that don't have matching parent items in pdbx_poly_seq_scheme
Warning, the input file is not valid. Run with --verbose to see why.

  warnings.warn(err)


Processed 8D05_A — 60 residues
Downloaded 6JEC
No valid chain found for 6JEC
Failed to download 10PX

Saved 599 clean chains → protein_sequences_ss_3.csv
